XGBoost
Antes de executar, Execute no terminal: py -m pip install xgboost 

In [3]:
import pandas as pd
from constantes import (
    ENDERECO_DATASET_PENSE,
    ENDERECO_DICIONARIO_MICRODADOS_PENSE,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS_DEPENDENTES,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS_INDEPENDENTES,
    VARIAVEIS_DEPENDENTES,
    VARIAVEIS_INDEPENDENTES,
    NUMERO_CLASSES_POR_VARIAVEL_DEPENDENTE,
    NOVA_LINHA,
    COLUNA_PESO
)

In [4]:
#  Carregar o dataset normalizado
caminho = "../dados/em_processamento/dados_preparados_v2.txt"
dados_normalizados = pd.read_csv(caminho, sep=";", low_memory=False)

In [5]:
# Verifica se todas as features constantes existem no dataframe
faltantes = [f for f in VARIAVEIS_INDEPENDENTES if f not in dados_normalizados.columns]
if faltantes:
    raise ValueError(f"As seguintes features constantes não foram encontradas na base: {faltantes}")

In [6]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#COLUNA_PESO= 'PESO_ALUNO_FREQ' # Importar de constantes
# Passar os vetores de constantes para extrair as matrizes X e y do DataFrame
X = dados_normalizados[VARIAVEIS_INDEPENDENTES]
y_multi = dados_normalizados[VARIAVEIS_DEPENDENTES]
pesos = dados_normalizados[COLUNA_PESO]


# DIVISÃO EM TREINAMENTO, VALIDAÇÃO E TESTE
# ==========================================
# Passo A: Separar 70% para Treino e 30% temporário (que será dividido em Validação e Teste)
X_treino, X_temp, y_treino, y_temp, pesos_treino, pesos_temp = train_test_split(
    X, y_multi, pesos, test_size=0.30, random_state=42
)

# Passo B: Dividir os 30% temporários igualmente em Validação (15%) e Teste (15%)
X_val, X_teste, y_val, y_teste, pesos_val, pesos_teste = train_test_split(
    X_temp, y_temp, pesos_temp, test_size=0.50, random_state=42
)

print(f"Tamanho do conjunto de Treino: {len(X_treino)} linhas")
print(f"Tamanho do conjunto de Validação: {len(X_val)} linhas")
print(f"Tamanho do conjunto de Teste: {len(X_teste)} linhas\n")
print("---Shapes das matrizes---")
print("Treinamento:", X_treino.shape, y_treino.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_teste.shape, y_teste.shape)


for target in VARIAVEIS_DEPENDENTES:
    print(f"\nTarget: {target}")
    print("Distribuição das classes:")
    print(y_treino[target].value_counts().sort_index())

Tamanho do conjunto de Treino: 87424 linhas
Tamanho do conjunto de Validação: 18734 linhas
Tamanho do conjunto de Teste: 18734 linhas

---Shapes das matrizes---
Treinamento: (87424, 123) (87424, 6)
Validação: (18734, 123) (18734, 6)
Teste: (18734, 123) (18734, 6)

Target: B12004
Distribuição das classes:
B12004
0.0      527
1.0     8083
2.0     9650
3.0    21449
4.0    24805
5.0    22910
Name: count, dtype: int64

Target: B12005
Distribuição das classes:
B12005
0.0      465
1.0     9374
2.0    20407
3.0    31792
4.0    17179
5.0     8207
Name: count, dtype: int64

Target: B12006
Distribuição das classes:
B12006
0.0      528
1.0    23558
2.0    19473
3.0    21588
4.0    12651
5.0     9626
Name: count, dtype: int64

Target: B12007
Distribuição das classes:
B12007
0.0      513
1.0     7761
2.0    13335
3.0    27757
4.0    22172
5.0    15886
Name: count, dtype: int64

Target: B12008
Distribuição das classes:
B12008
0.0      576
1.0    41917
2.0    15386
3.0    13946
4.0     7811
5.0     77

In [ ]:
#XGBoost treina um modelo para cada target, então vamos criar um dicionário para armazenar os modelos treinados e uma matriz para armazenar as predições no conjunto de teste.  
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
# Dicionário para armazenar os modelos treinados de cada target
modelos_treinados = {}   # nome do alvo -> modelo
encoders = {}            # nome do alvo -> LabelEncoder (para reverter as classes)
# Matriz para armazenar as predições no conjunto de teste (linhas x quantidade de targets)
y_pred = np.zeros(y_teste.shape, dtype=float)

# Pesos como arrays NumPy 1D
w_treino_arr = np.asarray(pesos_treino).ravel()
# Garante que os pesos sejam arrays NumPy unidimensionais
w_val_arr = np.asarray(pesos_val).ravel()

for i, nome_alvo in enumerate(y_treino.columns):
    # Classes -> 0..n-1 (obrigatório no XGBoost)
    le = LabelEncoder().fit(y_treino[nome_alvo])
    y_tr = le.transform(y_treino[nome_alvo])
    y_va = le.transform(y_val[nome_alvo])   # falha se a validação tiver classe ausente no treino
    print(f"Treinando modelo para o alvo: {nome_alvo} ({len(le.classes_)} classes)")
  # Instancia o XGBoost
    modelo = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",          # num_class é inferido dos dados
        eval_metric=["merror", "mlogloss"],  # Métricas avaliadas na validação: o early stopping usa a ÚLTIMA (mlogloss)
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
  # Treina usando a coluna de pesos específica da linha (sample_weight)
    modelo.fit(
        X_treino, y_tr,
        sample_weight=w_treino_arr,
        eval_set=[(X_val, y_va)],
        sample_weight_eval_set=[w_val_arr],
        verbose=False,
    )
  # Salva o modelo treinado no dicionário
    modelos_treinados[nome_alvo] = modelo
    encoders[nome_alvo] = le
    
    # Armazena as predições para o conjunto de teste
    # volta aos códigos originais (ex.: 1..5)
    y_pred[:, i] = le.inverse_transform(modelo.predict(X_teste))

    print(f"Melhor iteração para {nome_alvo}: {modelo.best_iteration}")

print("\nTreinamento de todos os alvos concluído.")

Treinando modelo para o alvo: B12004 (6 classes)
Melhor iteração para B12004: 250
Treinando modelo para o alvo: B12005 (6 classes)
Melhor iteração para B12005: 201
Treinando modelo para o alvo: B12006 (6 classes)
Melhor iteração para B12006: 159
Treinando modelo para o alvo: B12007 (6 classes)
Melhor iteração para B12007: 206
Treinando modelo para o alvo: B12008 (6 classes)


In [11]:
#XGBoost múltiplos targets - teste
from sklearn.multioutput import MultiOutputClassifier
#Normalizar targets


# Configurar e treinar o modelo utilizando os vetores
xgb_base = XGBClassifier(
    n_estimators=100, # Limite máximo de arvores
    learning_rate=0.05, 
    max_depth=5, 
    random_state=42,
    early_stopping_rounds=30, # Pinterrompe o treinamento se não houver melhora na métrica de validação apór 30 rodadas
    eval_metric=["logloss", "auc"]
)

# Envolve para aceitar múltiplos targets (Multi-label)
modelo_multialvo = MultiOutputClassifier(xgb_base)

# Treinamento passando as features, múltiplos alvos e o vetor de pesos por linha
modelo_multialvo.fit(X_treino, y_treino, sample_weight=pesos_treino)

print("Modelo treinado com sucesso!\n")

print("Melhor iteração:", modelo_multialvo.best_iteration)

ValueError: Must have at least 1 validation dataset for early stopping.

In [2]:
#Avaliação por alvo
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             log_loss, confusion_matrix, classification_report,
                             cohen_kappa_score, mean_absolute_error)

w_teste_arr = np.asarray(pesos_teste).ravel()
resultados = []
matrizes = {}

for i, nome_alvo in enumerate(y_teste.columns):
    modelo = modelos_treinados[nome_alvo]
    le = encoders[nome_alvo]

    y_real = y_teste[nome_alvo].to_numpy()
    y_prev = y_pred[:, i].astype(y_real.dtype)       # já nos códigos originais (ex.: 1..5)
    y_prob = modelo.predict_proba(X_teste)           # colunas = le.classes_

    # Classes do teste que o modelo nunca viu no treino (raro, mas quebra as métricas)
    desconhecidas = ~np.isin(y_real, le.classes_)
    if desconhecidas.any():
        print(f"[aviso] {nome_alvo}: {desconhecidas.sum()} linhas do teste com classe inexistente no treino; removidas.")
    m = ~desconhecidas

    # Para alvos ordinais (escalas 1-5), o erro importa pela distância; para nominais, ignore QWK/MAE
    resultados.append({
        "alvo": nome_alvo,
        "n_teste": int(m.sum()),
        "acuracia": accuracy_score(y_real[m], y_prev[m], sample_weight=w_teste_arr[m]),
        "acuracia_balanceada": balanced_accuracy_score(y_real[m], y_prev[m], sample_weight=w_teste_arr[m]),
        "f1_macro": f1_score(y_real[m], y_prev[m], average="macro", sample_weight=w_teste_arr[m], zero_division=0),
        "f1_ponderado": f1_score(y_real[m], y_prev[m], average="weighted", sample_weight=w_teste_arr[m], zero_division=0),
        "logloss": log_loss(y_real[m], y_prob[m], labels=le.classes_, sample_weight=w_teste_arr[m]),
        "kappa_quadratico": cohen_kappa_score(y_real[m], y_prev[m], weights="quadratic", sample_weight=w_teste_arr[m]),
        "mae": mean_absolute_error(y_real[m], y_prev[m], sample_weight=w_teste_arr[m]),
    })

    matrizes[nome_alvo] = pd.DataFrame(
        confusion_matrix(y_real[m], y_prev[m], labels=le.classes_, sample_weight=w_teste_arr[m]),
        index=[f"real_{c}" for c in le.classes_],
        columns=[f"prev_{c}" for c in le.classes_],
    )

tabela = pd.DataFrame(resultados).set_index("alvo").round(3)
print(tabela)

# Detalhe de um alvo: precisão, recall e F1 por classe
alvo = y_teste.columns[0]
print(classification_report(y_teste[alvo], y_pred[:, 0].astype(int), sample_weight=w_teste_arr, zero_division=0))
print(matrizes[alvo].round(0))

NameError: name 'pesos_teste' is not defined

In [7]:
#Comparação com um baseline
from sklearn.dummy import DummyClassifier

for i, nome_alvo in enumerate(y_teste.columns):
    base = DummyClassifier(strategy="prior").fit(X_treino, y_treino[nome_alvo], sample_weight=w_treino_arr)
    prev_base = base.predict(X_teste)
    print(nome_alvo,
          "| acc modelo:", round(accuracy_score(y_teste[nome_alvo], y_pred[:, i].astype(int), sample_weight=w_teste_arr), 3),
          "| acc baseline:", round(accuracy_score(y_teste[nome_alvo], prev_base, sample_weight=w_teste_arr), 3))

B12004 | acc modelo: 0.37 | acc baseline: 0.256
B12005 | acc modelo: 0.447 | acc baseline: 0.342
B12006 | acc modelo: 0.402 | acc baseline: 0.273
B12007 | acc modelo: 0.386 | acc baseline: 0.295
B12008 | acc modelo: 0.53 | acc baseline: 0.473
B12009 | acc modelo: 0.755 | acc baseline: 0.675


In [ ]:
#exibição das métricas de avaliação para cada target
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
#  AVALIAÇÃO DE MÉTRICAS (Conjunto de Teste)
# ==========================================
y_pred = modelo_multialvo.predict(X_teste)
y_proba = modelo_multialvo.predict_proba(X_teste) # Retorna lista de arrays por target

resultados_metricas = []

for i, target in enumerate(VARIAVEIS_INDEPENDENTES):
    # Extrai previsões e valores reais para o alvo atual
    y_true_col = y_teste.iloc[:, i]
    y_pred_col = y_pred[:, i]
    
    # Probabilidades da classe positiva (coluna 1) para calcular a perda (log_loss)
    # y_proba[i] tem shape (n_amostras, 2)
    y_prob_col = y_proba[i][:, 1] if y_proba[i].shape[1] > 1 else y_proba[i][:, 0]

    # Cálculo das métricas
    acuracia = accuracy_score(y_true_col, y_pred_col)
    f1 = f1_score(y_true_col, y_pred_col, average='weighted', zero_division=0)
    perda = log_loss(y_true_col, y_prob_col)
    
    resultados_metricas.append({
        'Target': target,
        'Acuracia': acuracia,
        'F1_Score': f1,
        'Perda_LogLoss': perda
    })

df_metricas = pd.DataFrame(resultados_metricas)
print("--- Tabela de Métricas por Alvo ---")
print(df_metricas.to_string(index=False))

# ==========================================
#  PLOTAGEM DAS MÉTRICAS
# ==========================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 5))

# Gráfico 1: Acurácia e F1-Score por Alvo
plt.subplot(1, 2, 1)
df_melted = df_metricas.melt(id_vars='Target', value_vars=['Acuracia', 'F1_Score'], 
                             var_name='Metrica', value_name='Valor')
sns.barplot(data=df_melted, x='Target', y='Valor', hue='Metrica', palette='viridis')
plt.title('Comparativo de Acurácia e F1-Score por Alvo')
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.xlabel('Variáveis Alvo')
plt.ylabel('Pontuação')

# Gráfico 2: Perda (Log Loss) por Alvo
plt.subplot(1, 2, 2)
sns.barplot(data=df_metricas, x='Target', y='Perda_LogLoss', palette='magma')
plt.title('Perda (Log Loss) por Alvo (Menor é melhor)')
plt.xticks(rotation=15)
plt.xlabel('Variáveis Alvo')
plt.ylabel('Log Loss')

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'seaborn'